# W9 Practice — LangChain Core Patterns

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week09/W9_practice_langchain_core.ipynb)

**Goal.** Learn LangChain by writing the patterns of the Week 5 practice notebook (v2) again: one call, a conversation, structured output, a tool call by hand, and the agent loop. Most examples are the same as in Week 5, so you can compare each LangChain line with the OpenAI SDK line that you already know. At the end, `create_agent` runs the whole loop in one call.

**LangChain** is an open-source framework for applications that use language models. It gives one interface for chat models from many providers, classes for messages and tools, and ready-made agents. Its agents run on **LangGraph**, a library that runs a program as a graph: steps connected by arrows, which can form a loop (Part 6).

| Part | Week 5 (OpenAI SDK) | This notebook (LangChain) |
|---|---|---|
| 1. One call | `client.chat.completions.create(...)` | `model.invoke(...)` |
| 2. A conversation | message dictionaries | message objects |
| 3. Structured output | `response_format={"type": "json_object"}` + `json.loads` | `with_structured_output` |
| 4. A tool call by hand | `tool_from_function`, `"tool_calls" in reply` | `@tool`, `bind_tools`, `reply.tool_calls` |
| 5. The agent loop | `run_loop` | the same loop with LangChain objects |
| 6. The ready-made agent | your `Agent` class (Week 5 Part 7) | `create_agent` |

Week 5 Part 6 (ReAct) is left out here. The Week 9 lab uses Parts 1 and 2; later weeks use Parts 3 to 6.

**Cell labels.** The first line of each code cell says what to do with it:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.

It takes about 60 minutes and is not collected. Do this notebook before the Week 9 lab.

## 0. Setup

*Do:* run the three cells; paste your API key into the second. The third cell makes the model object: `init_chat_model("openai:gpt-4.1")` replaces `client = OpenAI()` and `MODEL` of Week 5, and the part before the colon names the provider. The cell also defines `show_messages`, which prints each message: its position, its type (its role, see Part 2), and its content, or for a request, the requested tool and its arguments.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14"

In [ ]:
# ▶ RUN · setup: just run
import os

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

In [ ]:
# ▶ RUN · setup: just run
import json
import requests
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

MODEL = "openai:gpt-4.1"           # "provider:model"; the prefix tells LangChain which client to use
model = init_chat_model(MODEL)     # one model object for the whole notebook

def show_messages(messages):
    """Print each message: position, type, and content (or the requested tool and its arguments)."""
    for i in range(len(messages)):
        m = messages[i]
        if m.type == "ai" and m.tool_calls:            # a request: print each requested tool
            for request in m.tool_calls:
                print(i, m.type, request["name"], request["args"])
        else:
            print(i, m.type, m.content)
    print()

## 1. One Call

In Week 5, one call was three lines: `client.chat.completions.create(...)`, then `model_dump` to get a dictionary, then `append`. In LangChain, `model.invoke(...)` replaces the first two lines: it sends the messages and returns the reply as an `AIMessage` **object**. Keeping the reply with `append` is still your job (Part 2). `invoke` also takes a plain string and sends it as one user message.

You read the parts of an object as attributes, with a dot: `reply.content` instead of `reply["content"]`.

### 1.1 One call

*Do:* run the cell. It prints the class of the reply, its content, and the number of tokens that the call used.

In [ ]:
# ▶ RUN · example: read each line, then run
QUESTION = "In one sentence, what is a canceled order in an online shop?"
reply = model.invoke(QUESTION)                     # a string is sent as one user message
print(type(reply).__name__)                        # the name of the class of the reply
print(reply.content)
usage = reply.usage_metadata                       # a dictionary of token counts
print("input tokens:", usage["input_tokens"], "| output tokens:", usage["output_tokens"])

**Reading the output.** `AIMessage` is the LangChain class for an assistant message. `reply.content` is the text, as `reply["content"]` was in Week 5. `reply.usage_metadata` is a plain dictionary, so you read it with brackets: `input_tokens`, `output_tokens`, and `total_tokens`. The Week 9 lab reads `input_tokens` to compare the cost of different inputs.

### 1.2 A first call

*Do:* run the cell. It sends `"Reply with exactly: PASS"` to the model. Expected: `PASS`.

In [ ]:
# ▶ RUN · example: read each line, then run
reply = model.invoke("Reply with exactly: PASS")
print(reply.content)

## 2. A Conversation: the List Grows

A list of messages works as in Week 5: you add the messages that you write, and you keep each reply. Each role has a class:

| Week 5 dictionary | LangChain object | `.type` |
|---|---|---|
| `{"role": "system", "content": ...}` | `SystemMessage(...)` | `system` |
| `{"role": "user", "content": ...}` | `HumanMessage(...)` | `human` |
| the reply, `{"role": "assistant", ...}` | `AIMessage` (returned by `invoke`) | `ai` |
| `{"role": "tool", "tool_call_id": ..., ...}` | `ToolMessage(...)` (Part 4) | `tool` |

LangChain calls the user role `human` and the assistant role `ai`; `.type` holds this name. `invoke` also accepts the dictionaries of Week 5, but use the objects here: the check cells read `.type`.

### 2.1 Two turns

*Do:* run the four cells one by one. Units 2 and 4 are the same code: call the model with the whole list and keep the reply.

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 1 - the list: a system message and a user message
STAFF_SYSTEM = "You answer questions from shop staff. Answer in at most two sentences."
QUESTION = "A customer ordered 2 mugs and 1 desk lamp. List the items in the order."

messages = [
    SystemMessage(STAFF_SYSTEM),                     # system message
    HumanMessage(QUESTION),                          # human (user) message
]
show_messages(messages)

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 2 - call the model, keep the reply
reply = model.invoke(messages)
messages.append(reply)                               # ai (assistant) message
show_messages(messages)

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 3 - add the next question
messages.append(HumanMessage("The lamp was canceled. What is left in the order?"))   # human (user) message
show_messages(messages)

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 4 - call the model, keep the reply (the same code as unit 2)
reply = model.invoke(messages)
messages.append(reply)                               # ai (assistant) message
show_messages(messages)

**Reading the output.** The list grows 2 → 3 → 4 → 5. The model answers the second question correctly only because the list still holds the first question and its answer: the model itself keeps nothing between calls, as in Week 5.

### 2.2 One more turn

*Do:* run the cell. It adds the question `"The customer adds 3 notebooks. What is in the order now?"`, calls the model, and keeps the reply: three lines, as in units 3 and 4. Expected: 7 messages, and an answer that names the mugs and the notebooks. If you run the cell twice, the list grows twice; then run 2.1 again from unit 1.

In [ ]:
# ▶ RUN · example: read each line, then run
messages.append(HumanMessage("The customer adds 3 notebooks. What is in the order now?"))
reply = model.invoke(messages)
messages.append(reply)
show_messages(messages)

## 3. Structured Output: an Object instead of JSON Text

In Week 5 you described the JSON keys in the prompt, set `response_format={"type": "json_object"}`, and read the text with `json.loads`. In LangChain you describe the result as a class instead.

The class is written with **Pydantic**, a library installed with LangChain for classes whose fields have fixed types. `(BaseModel)` after the class name makes it a Pydantic class. Each line `name: type` declares one field, in the same notation as `city: str` in a function. The two classes describe the JSON of Week 5: `Item` has `product` and `quantity`, and `Order` has `items` (`list[Item]`, a list of `Item` objects) and `city`.

`model.with_structured_output(Order)` returns a new model. It sends the class as a JSON schema with each call, so the API returns exactly these fields, and LangChain reads the JSON into an `Order` object.

### 3.1 An order as an object

*Do:* run the cell. It prints the object, then reads its fields with a dot.

In [ ]:
# ▶ RUN · example: read each line, then run
from pydantic import BaseModel

class Item(BaseModel):
    product: str
    quantity: int

class Order(BaseModel):
    items: list[Item]                              # a list of Item objects
    city: str

order_model = model.with_structured_output(Order)  # replies of this model are Order objects
customer_message = "Hi, I'd like to order 3 notebooks and 2 pen sets, delivered to Busan."
order = order_model.invoke(f"""Extract the order from the customer message.

Customer message: {customer_message}""")
print(order)
for item in order.items:
    print(item.product, item.quantity)
print(order.city)

**Reading the output.** The prompt says only what to extract; the keys come from the class, so the prompt no longer lists them. `order.items` is a list of `Item` objects. `item.quantity` is an `int` because Pydantic checked the type. You read the fields with a dot, `order.city`, where Week 5 used `order["city"]`.

### 3.2 Another order

*Do:* run the cell. It calls `order_model` for the message below, with the prompt of 3.1, and adds up the quantities of the items in `total_quantity`. Expected: Seoul and 5.

In [ ]:
# ▶ RUN · example: read each line, then run
customer_message = "Please send 1 desk lamp and 4 mugs to Seoul."
order = order_model.invoke(f"""Extract the order from the customer message.

Customer message: {customer_message}""")
total_quantity = 0
for item in order.items:
    total_quantity = total_quantity + item.quantity
print(order)
print(total_quantity)

## 4. A Tool Call by Hand

The round trip is the same as in Weeks 3 to 5: the model requests a tool, your code runs it, and a tool message carries the result back with the id of the request.

### 4.1 The tool

In Week 5 a tool was two things: the function in `FUNCTIONS` and its schema in `TOOLS`, made by `tool_from_function`. In LangChain one **tool object** holds both. `@tool` on the line above `def` is a **decorator**: Python passes the function to `tool` and stores the result under the same name. So `get_weather` is now a tool object, not a plain function. `get_weather("Seoul")` no longer works; `get_weather.invoke({"city": "Seoul"})` runs it, with the arguments as one dictionary.

The schema comes from the name, the docstring, and the type annotations, as with `tool_from_function`; `parse_docstring=True` also reads the `Args:` lines. The body is the Week 5 body with one change: it returns JSON text (`json.dumps`), because a tool message holds text.

*Do:* run the cell. It prints the schema that the model will see, in the format of `TOOLS[0]` in Week 5 (`convert_to_openai_tool` writes it out), then runs the tool once by hand.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """Get the current temperature of a city in Celsius.

    Args:
        city: City name in English, for example "Seoul".
    """
    try:                                           # on a network or lookup error, return the error as text
        place = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                             params={"name": city, "count": 1}, timeout=10).json()["results"][0]   # first match
        weather = requests.get("https://api.open-meteo.com/v1/forecast",
                               params={"latitude": place["latitude"], "longitude": place["longitude"],
                                       "current": "temperature_2m"}, timeout=10).json()
        return json.dumps({"city": place["name"], "temperature_c": weather["current"]["temperature_2m"]})
    except Exception as error:
        return "error: " + str(error)

print(json.dumps(convert_to_openai_tool(get_weather), indent=2))   # the schema, as TOOLS[0] in Week 5
print(get_weather.invoke({"city": "Seoul"}))                        # run the tool: the arguments as one dictionary

**Reading the output.** The schema has the same keys as `TOOLS[0]` in Week 5: the name, the description, and the parameter `city` with its description. If Open-Meteo cannot be reached, the tool returns the error as text, and you can continue.

### 4.2 Call, check, call, check

`model.bind_tools([get_weather])` returns a new model that sends the tool schema with every call, as `tools=TOOLS` did in Week 5; `model` itself stays without tools. Every reply is an `AIMessage`. Week 5 checked whether `reply` had the key `"tool_calls"`; now you check whether the list `reply.tool_calls` is empty. For a request, the list holds one dictionary per request, with `name`, `args`, `id`, and `type`. `args` is already a dictionary, so `json.loads` and `**args` are not needed. For a text answer, the list is `[]`, and the text is in `reply.content`. As in Week 5, every request needs its tool message before the next call.

*Do:* run the five cells in order and watch the list grow 2 → 3 → 4 → 5. Units 2 and 4 are the same code, and so are units 3 and 5. If you run a cell twice, start again from unit 1.

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 1 - the list, and a model that knows the tool
WEATHER_SYSTEM = "You answer questions about the weather. Use the tools for current values; never guess them."
QUESTION = "What is the temperature in Seoul right now?"

weather_model = model.bind_tools([get_weather])    # calls of weather_model include the tool schema
messages = [SystemMessage(WEATHER_SYSTEM), HumanMessage(QUESTION)]
answer = None
show_messages(messages)

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 2 - call the model
reply = weather_model.invoke(messages)
messages.append(reply)                               # ai (assistant) message
print(reply.tool_calls)                              # the requests: name, args, id, type
print(repr(reply.content))                           # the text of the reply; repr shows an empty text as ''
show_messages(messages)

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 3 - check: run each requested tool, or read the answer
## (one tool here, so get_weather runs directly; 5.2 finds each tool by its name)
if reply.tool_calls:                                 # a non-empty list counts as True
    for request in reply.tool_calls:
        result = get_weather.invoke(request["args"])
        messages.append(ToolMessage(result, tool_call_id=request["id"]))   # TOOL message
else:
    answer = reply.content
show_messages(messages)

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 4 - call the model (the same code as unit 2)
reply = weather_model.invoke(messages)
messages.append(reply)                               # ai (assistant) message
print(reply.tool_calls)
show_messages(messages)

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 5 - check (the same code as unit 3)
if reply.tool_calls:
    for request in reply.tool_calls:
        result = get_weather.invoke(request["args"])
        messages.append(ToolMessage(result, tool_call_id=request["id"]))
else:
    answer = reply.content
print("answer:", answer)

**Reading the output.** The first reply has a request, and its content is empty (`''`). The second reply has no requests (`[]`) and holds the answer. `ToolMessage(result, tool_call_id=...)` is the tool message of Week 5, `{"role": "tool", "tool_call_id": ..., "content": ...}`, written as an object.

### 4.3 Run the tool for Tokyo

*Do:* run the cell. It builds the list for Tokyo, calls the model once, and runs the loop: for each request, it runs the tool and adds the tool message (two lines, as in unit 3). Then it prints the list; the last message is a tool message. The next call would give the answer, as in unit 4.

In [ ]:
# ▶ RUN · example: read each line, then run
messages = [SystemMessage(WEATHER_SYSTEM), HumanMessage("What is the temperature in Tokyo right now?")]
reply = weather_model.invoke(messages)
messages.append(reply)
for request in reply.tool_calls:
    result = get_weather.invoke(request["args"])
    messages.append(ToolMessage(result, tool_call_id=request["id"]))
show_messages(messages)

## 5. The Agent Loop

The loop repeats call and check until the reply has no requests, as `run_loop` did in Week 5.

### 5.1 A calculator tool

This is the Week 5 `calculate` as a LangChain tool. Like `get_weather`, it returns text (`str(...)`), because a tool message holds text. `eval` runs the text as Python; `{"__builtins__": {}}` removes the built-in functions, which is enough for this practice but not a real safeguard (Chapter 14).

*Do:* run the cell.

In [ ]:
# ▶ RUN · example: read each line, then run
@tool(parse_docstring=True)
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression and return the result.

    Args:
        expression: A Python arithmetic expression, for example "58 * 94 + 37 * 21".
    """
    try:
        return str(eval(expression, {"__builtins__": {}}))   # run the text as Python, with no built-in functions
    except Exception as error:
        return "error: " + str(error)

print(calculate.invoke({"expression": "58 * 94 + 37 * 21"}))

### 5.2 The loop as a function

`run_loop` is the Week 5 function with LangChain objects. Two things changed. There is no `functions` argument: each tool object carries its name and its function, so the dictionary `tools_by_name`, built inside, replaces `FUNCTIONS`. And `model.bind_tools(tools)` replaces `tools=tools` in the call.

*Do:* run the cell. It defines `run_loop` and runs it with both tools on the Week 5 question about Seoul in Fahrenheit.

In [ ]:
# ▶ RUN · example: read each line, then run
def run_loop(messages, tools, max_steps=6):
    """Call the model, run the requested tools, and repeat until the reply has no requests."""
    tools_by_name = {}
    for t in tools:
        tools_by_name[t.name] = t                  # name -> tool object
    loop_model = model.bind_tools(tools)
    for step in range(max_steps):
        reply = loop_model.invoke(messages)
        messages.append(reply)
        if reply.tool_calls:
            for request in reply.tool_calls:
                result = tools_by_name[request["name"]].invoke(request["args"])
                messages.append(ToolMessage(result, tool_call_id=request["id"]))
        else:
            return reply.content
    return None                                    # the step limit ended the loop

LOOP_SYSTEM = "You answer questions with your tools. Use calculate for every calculation."
QUESTION = "What is the current temperature in Seoul in Fahrenheit? Use F = C × 1.8 + 32."
messages = [SystemMessage(LOOP_SYSTEM), HumanMessage(QUESTION)]
answer = run_loop(messages, [get_weather, calculate])
show_messages(messages)
print("answer:", answer)

**Reading the output.** In our runs, the model asked for the weather first, then for the calculation, and then answered: each reply waited for the result of the one before. One reply can also request several tools at once. Then the requests share one position in the printed list, and the loop adds one tool message for each request. In both cases, the loop calls the model again until a reply has no requests.

## 6. The Ready-Made Agent: `create_agent`

`create_agent` builds the loop of 5.2 for you. It takes the model, the tools, and the system prompt, as your `Agent` class did, and returns an agent. The agent keeps its data in a dictionary, the **state**; the message list is under the key `"messages"`. `invoke` takes a state with your question and returns the final state. Unlike your `Agent` class, the agent does not keep the list between calls: for a follow-up question, pass `state["messages"]` with the new question added.

### 6.1 The same question

*Do:* run the cell and compare the list with the list of 5.2: usually the same types in the same order. The system prompt is not in the returned list, because the agent adds it to each model call, so each position is one lower than in 5.2. The last lines print the graph of the agent: each arrow from one step to the next.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.agents import create_agent

agent = create_agent(model, tools=[get_weather, calculate], system_prompt=LOOP_SYSTEM)
state = agent.invoke({"messages": [HumanMessage(QUESTION)]})
show_messages(state["messages"])
for edge in agent.get_graph().edges:               # each arrow from one step to the next
    print(edge.source, "->", edge.target)

**Reading the output.** Each step of the agent is a **node**, and each arrow is an **edge**. `__start__` and `__end__` mark where a run begins and ends. After `model`, the graph goes to `tools` if the reply has requests, and to `__end__` if not; after `tools`, it goes back to `model`. This is the `for` loop and the `if reply.tool_calls:` of `run_loop`.

**Next.** The Week 9 lab uses Parts 1 and 2 (`init_chat_model`, `invoke`, message objects, `usage_metadata`). It builds a RAG program over the EU AI Act without an agent. In Week 10, the search of that program becomes a tool (Part 4) of a `create_agent` agent (Part 6).